<a href="https://colab.research.google.com/github/IGol22/SFML/blob/main/%D0%9A%D1%83%D1%80%D1%81%D0%BE%D0%B2%D0%B0%D1%8F/7_Classification_SI_threshold8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q catboost xgboost

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier, HistGradientBoostingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Подгрузка датасета с GitHub
file_name = "dataset_classic_ML.xlsx"
if not os.path.exists(file_name):
    raw_url = "https://raw.githubusercontent.com/IGol22/SFML/main/Курсовая/dataset_classic_ML.xlsx"
    !wget -q -O {file_name} "{raw_url}"

df = pd.read_excel(file_name)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00


,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,...,0,0,0,0,0,0,0,0,3,0
1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,...,0,0,0,0,0,0,0,0,3,0
2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,446.808,...,0,0,0,0,0,0,0,0,3,0
3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,398.679,...,0,0,0,0,0,0,0,0,4,0
4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,466.713,...,0,0,0,0,0,0,0,0,0,0


In [2]:
# 1. Формируем таргет SI с порогом 8
target_col = 'SI'
targets_to_drop = ['IC50, mM', 'CC50, mM', 'SI']

X = df.drop(columns=targets_to_drop, errors='ignore').copy()

# Деление по фиксированному порогу: 1 если SI > 8, иначе 0
threshold_val = 8.0
y = (df[target_col] > threshold_val).astype(int)

print(f"Порог SI: {threshold_val}")
print(f"Распределение классов:\n{y.value_counts()}")

# 2. Feature Engineering
if 'MolLogP' in X.columns and 'MolWt' in X.columns:
    X['MolLogP_x_MolWt'] = X['MolLogP'] * X['MolWt']

poly_cols = [c for c in ['MolLogP', 'MolWt'] if c in X.columns]
if poly_cols:
    poly = PolynomialFeatures(degree=2, include_bias=False)
    poly_feats = poly.fit_transform(X[poly_cols])
    poly_df = pd.DataFrame(poly_feats, columns=poly.get_feature_names_out(poly_cols), index=X.index)
    for col in poly_df.columns:
        if col not in X.columns:
            X[col] = poly_df[col]

if 'MolLogP' in X.columns:
    X['MolLogP_gt_3'] = (X['MolLogP'] > 3).astype(int)

# 3. Заполнение пропусков
if X.isnull().values.any():
    imputer = SimpleImputer(strategy='median')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print(f"Размерность матрицы X: {X.shape}, Длина y: {len(y)}")

Порог SI: 8.0
Распределение классов:
SI
0    644
1    357
Name: count, dtype: int64
Размерность матрицы X: (1001, 215), Длина y: 1001


In [3]:
# Разбиение с сохранением баланса классов (stratify)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Отбор признаков и масштабирование
vt = VarianceThreshold(threshold=0.01)
X_train_sel = vt.fit_transform(X_train)
X_test_sel = vt.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

# Модели
models = {
    'KNN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0),
    'Stacking': StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(random_state=42)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('xgb', XGBClassifier(random_state=42, eval_metric='logloss'))
        ],
        final_estimator=LogisticRegression()
    )
}

# Обучение
results = []
for name, model in models.items():
    tr_x = X_train_scaled if name in ['KNN'] else X_train_sel
    te_x = X_test_scaled if name in ['KNN'] else X_test_sel

    model.fit(tr_x, y_train)
    y_pred = model.predict(te_x)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(te_x)[:, 1]
    else:
        y_prob = y_pred

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    results.append({'Model': name, 'Accuracy': acc, 'F1': f1, 'ROC-AUC': roc_auc})

res_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False).round(3)
res_df

,Model,Accuracy,F1,ROC-AUC
0,KNN,0.726,0.615,0.753
6,CatBoost,0.726,0.574,0.747
7,Stacking,0.726,0.553,0.737
2,Gradient Boosting,0.716,0.558,0.734
1,Random Forest,0.706,0.556,0.731
4,AdaBoost,0.706,0.528,0.717
3,HistGradientBoosting,0.692,0.544,0.715
5,XGBoost,0.711,0.580,0.712


## Выводы и рекомендации

### Сравнение моделей
* **Лидер по качеству:** Наилучшее качество ранжирования показал алгоритм **KNN** (ROC-AUC = 0.753, Accuracy = 0.726, F1 = 0.615). На втором месте — **CatBoost** (ROC-AUC = 0.747).
* **Влияние дисбаланса классов:** Из-за несимметричного распределения (64% неактивных против 36% высокоселективных) значения F1-score закономерно снизились у всех моделей (0.528–0.615).
* **Сравнение с медианным сплитом:** Фиксированный биологический порог ($SI > 8$) дал более явный сигнал для моделей — итоговый ROC-AUC (0.71–0.75) оказался выше, чем при искусственном делении по медиане (0.66–0.70).

---

### Рекомендации
* **Выбор модели:** Для отбора соединений с высокой селективностью целесообразно использовать **KNN** или **CatBoost**.
* **Оптимизация при внедрении:** В практических задачах рекомендуется калибровать порог классификации (decision threshold) для минимизации ложноотрицательных результатов (чтобы случайно не отбросить действительно перспективное лекарство).